# Hacker News Pipeline

### Table of Contents<a id="toc"></a>

1. [Introduction and Setup](#intro)
2. [Loading Raw Data](#raw)
3. [Filtering for Signal](#signal)
4. [Tabular Transformation](#tabular)
5. [Title Extraction](#title)
6. [Text Normalization](#text)
7. [Building Keyword Frequencies](#keyword)
8. [Extracting Top Trends](#trends)
9. [Analyzing Pipeline Output](#analysis)
10. [Conclusion](#conclusion)

### Introduction and Setup
<a id="intro"></a>
Think of a data pipeline as a digital assembly line: raw, unorganized data enters on one end, passes through a series of specialized processing stations, and exits as polished, actionable intelligence. In this project, we process a dataset of 2014 Hacker News submissions to unearth the tech industry's top discussion topics. Using a custom pipeline framework, each task function handles a distinct stage of filtering, transforming, and analyzing the stream, building a seamless bridge from messy records to underlying industry trends.

In [1]:
from datetime import datetime
import json
import io
import csv

from pipeline import build_csv, Pipeline
from stop_words import stop_words

[Back to Table of Contents](#toc)

### Loading Raw Data (file_to_json)
<a id="raw"></a>
The journey begins by opening hn_stories_2014.json, parsing the raw JSON payload into native Python dictionaries, and isolating the primary list of story objects into memory.

In [2]:
pipeline = Pipeline()

@pipeline.task()
def file_to_json():
    with open('hn_stories_2014.json', 'r') as f:
        data = json.load(f)
        stories = data['stories']
    return stories

[Back to Table of Contents](#toc)

### Filtering for Signal (filter_stories)
<a id="signal"></a>
Once loaded, the stream passes through a gatekeeper function that discards low-engagement posts by retaining only stories with over 50 points and more than one comment, while stripping out "Ask HN" entries to focus exclusively on news items.

In [3]:
@pipeline.task(depends_on=file_to_json)
def filter_stories(stories):
    def is_popular(story):
        return story['points'] > 50 and story['num_comments'] > 1 and not story['title'].startswith('Ask HN')
    
    return (
        story for story in stories
        if is_popular(story)
    )

[Back to Table of Contents](#toc)

### Tabular Transformation (json_to_csv)
<a id="tabular"></a>
With noise eliminated, the filtered stories are converted into tabular rows; string timestamps are parsed into datetime objects, and the records are written directly to an in-memory io.StringIO buffer to avoid unnecessary disk I/O overhead.

In [4]:
@pipeline.task(depends_on=filter_stories)
def json_to_csv(stories):
    lines = []
    for story in stories:
        lines.append(
            (story['objectID'], datetime.strptime(story['created_at'], "%Y-%m-%dT%H:%M:%SZ"), story['url'], story['points'], story['title'])
        )
    return build_csv(lines, header=['objectID', 'created_at', 'url', 'points', 'title'], file=io.StringIO())

[Back to Table of Contents](#toc)

### Title Extraction (extract_titles)
<a id="title"></a>
Tapping into that in-memory CSV stream, the reader dynamically locates the title column index and streams headline strings downstream using a memory-efficient Python generator.

In [5]:
@pipeline.task(depends_on=json_to_csv)
def extract_titles(csv_file):
    reader = csv.reader(csv_file)
    header = next(reader)
    idx = header.index('title')
    
    return (line[idx] for line in reader)

[Back to Table of Contents](#toc)

### Text Normalization (clean_title)
<a id="text"></a>
To prepare the headlines for statistical text analysis, each title string is converted to lowercase and stripped of punctuation marks so variations like "Python!" and "python" are matched identically.

In [6]:
@pipeline.task(depends_on=extract_titles)
def clean_title(titles):
    for title in titles:
        title = title.lower()
        title = ''.join(c for c in title if c.isalnum() or c.isspace())
        yield title

[Back to Table of Contents](#toc)

### Building Keyword Frequencies (build_keyword_dictionary)
<a id="keyword"></a>
The normalized text is tokenized into individual words, cross-referenced against a stop_words list to drop generic filler words like "the" or "and", and tallied inside a key-value frequency dictionary.

In [7]:
@pipeline.task(depends_on=clean_title)
def build_keyword_dictionary(titles):
    word_freq = {}
    for title in titles:
        for word in title.split():
            if word and word not in stop_words:
                if word not in word_freq:
                    word_freq[word] = 1
                else:
                    word_freq[word] += 1
    return word_freq

[Back to Table of Contents](#toc)

### Extracting Top Trends (top_keywords)
<a id="trends"></a>
In the final processing station, the frequency dictionary is sorted in descending order to slice out the top 100 most influential keywords.

In [8]:
@pipeline.task(depends_on=build_keyword_dictionary)
def top_keywords(word_freq):
    freq_tuple = [
        (word, word_freq[word])
        for word in sorted(word_freq, key=word_freq.get, reverse=True)
    ]
    return freq_tuple[:100]

In [9]:
ran = pipeline.run()
print(ran[top_keywords])

[('new', 7), ('google', 4), ('pdf', 4), ('apple', 4), ('3', 4), ('software', 4), ('data', 4), ('web', 4), ('python', 4), ('git', 3), ('work', 3), ('release', 3), ('users', 3), ('support', 3), ('language', 3), ('history', 3), ('does', 3), ('just', 3), ('true', 2), ('truecrypt', 2), ('hire', 2), ('man', 2), ('20', 2), ('2013', 2), ('useast1', 2), ('app', 2), ('android', 2), ('introducing', 2), ('worth', 2), ('start', 2), ('internet', 2), ('developer', 2), ('time', 2), ('selfdriving', 2), ('nginx', 2), ('microsoft', 2), ('tech', 2), ('copyright', 2), ('generation', 2), ('use', 2), ('released', 2), ('server', 2), ('1', 2), ('vesper', 2), ('free', 2), ('analysis', 2), ('scroll', 2), ('building', 2), ('os', 2), ('check', 2), ('important', 2), ('homeless', 2), ('project', 2), ('human', 2), ('numbers', 2), ('tweets', 2), ('learning', 2), ('replaced', 2), ('linux', 2), ('platform', 2), ('http20', 2), ('website', 2), ('amazon', 2), ('goodbye', 1), ('using', 1), ('secure', 1), ('dedicated', 1), (

### Analyzing Pipeline Output
<a id="analysis"></a>
Executing the pipeline reveals an intriguing snapshot of the tech community's pulse in 2014. Unsurprisingly, the descriptor "new" tops the frequency list with 7 occurrences, emphasizing Hacker News' heavy focus on fresh product launches and updates. Major industry behemoths like "google" and "apple" sit in the upper tier (4 occurrences each) alongside foundational technical domains such as "software," "data," "web," and "python". Looking further down the list reveals specific technical events and infrastructure shifts from that era—including security chatter around "truecrypt," cloud operations markers like AWS's "useast1," mobile platforms ("android"), and early buzz surrounding "selfdriving" technology and "amazon".

### Conclusion
<a id="conclusion"></a>
By chaining these decoupled stages together, the pipeline seamlessly distills thousands of raw JSON entries into a clear snapshot of 2014's biggest tech trends—surfacing dominant industry terms like "google," "apple," "python," and "data". Placing each task inside its own notebook cell gives the project a clean, readable structure, keeping each transformation step modular, easy to debug, and simple to iterate on without re-executing the entire assembly line.

[Back to Table of Contents](#toc)